# Full-data baseline on a Kaggle GPU

Trains YOLOv8n on the whole training split with the same fixed protocol as the AL runs.

In [ ]:
!pip install -q ultralytics==8.4.51 scipy==1.17.1 pyyaml==6.0.3

In [ ]:
# Set these three values to match the two Kaggle Datasets added as input.
CODE_DIR   = "/kaggle/input/albench-code"
EXPORT_DIR = "/kaggle/input/<data-slug>/export_sortwaste"
CONFIG     = "sortwaste"

FRAC = 1.0       # 1.0 for the full-data baseline, 0.05 for a quick check
DEVICE = None    # for example "0,1" to train with DDP

In [ ]:
import json, os, sys
from pathlib import Path
import yaml

WORK = Path("/kaggle/working")
os.chdir(WORK)
CODE = Path(CODE_DIR)
EXPORT = Path(EXPORT_DIR).resolve()
assert (CODE / "albench").is_dir(), f"albench/ not found in {CODE}"
assert (CODE / "configs").is_dir(), f"configs/ not found in {CODE}"
assert (CODE / "yolov8n.pt").is_file(), "yolov8n.pt not found in the code dataset"
assert EXPORT.is_dir(), f"EXPORT_DIR does not exist: {EXPORT}"
os.environ["ALBENCH_WORK"] = str(WORK)
os.environ["DATA_ROOT"] = str(EXPORT.parent)
if DEVICE:
    os.environ["ALBENCH_DEVICE"] = DEVICE
sys.path.insert(0, str(CODE))

import scipy, torch, ultralytics
assert ultralytics.__version__ == "8.4.51"
assert scipy.__version__ == "1.17.1"
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings."
print("torch", torch.__version__, "CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

manifest_path = EXPORT.parent / f"{CONFIG}-data-manifest.json"
assert manifest_path.is_file(), f"manifest not found: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
expected = {item["path"]: int(item["bytes"]) for item in manifest["files"]}
actual = {p.relative_to(EXPORT.parent).as_posix(): p.stat().st_size
          for p in EXPORT.rglob("*") if p.is_file() and p.suffix != ".cache"}
missing = sorted(set(expected) - set(actual))
extra = sorted(set(actual) - set(expected))
wrong = sorted(k for k in set(expected) & set(actual) if expected[k] != actual[k])
assert not (missing or extra or wrong), (missing[:5], extra[:5], wrong[:5])
print("Dataset manifest OK:", len(actual), "files")

from albench.config import load_config, load_seeds
cfg = load_config(CONFIG)
cfg["baseline"]["cache"] = False
cfg["al"]["cache"] = False
cfg["baseline"]["model"] = cfg["al"]["model"] = str(CODE / "yolov8n.pt")
assert cfg["baseline"]["batch"] == 64, "the protocol fixes batch=64"
assert cfg["baseline"]["epochs"] == 100, "the protocol fixes 100 epochs"
assert cfg["baseline"]["warmup_epochs"] == 0.0, "the protocol fixes warmup_epochs=0"
assert cfg["baseline"]["close_mosaic"] == 0, "the protocol fixes close_mosaic=0"
assert cfg["baseline"]["eval_checkpoint"] == "last", "the protocol uses last.pt"
CFG_KAGGLE = WORK / "config_kaggle.yaml"
CFG_KAGGLE.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding="utf-8")
d = yaml.safe_load((EXPORT / "data.yaml").read_text(encoding="utf-8"))
d.update(path=str(EXPORT), nc=cfg["classes"]["nc"], names=cfg["classes"]["names"])
DATA_YAML = WORK / "data_kaggle.yaml"
DATA_YAML.write_text(yaml.safe_dump(d, allow_unicode=True), encoding="utf-8")
print("config ->", CFG_KAGGLE)

In [ ]:
import importlib.util
script = CODE / "scripts" / "05_train_baseline.py"
spec = importlib.util.spec_from_file_location("_baseline", str(script))
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
module.run(str(CFG_KAGGLE), None, str(DATA_YAML), FRAC)

In [ ]:
seed = load_seeds()[cfg["split"]["seed_index"]]
name = f"baseline_frac{FRAC}_seed{seed}"
summary = Path(cfg["reports_dir"]) / "baseline" / name / "SUMMARY.md"
assert summary.is_file(), f"{summary} not found"
print(summary.read_text(encoding="utf-8"))
weights = sorted((Path(cfg["baseline"]["project"]) / name / "weights").glob("*.pt"))
assert weights, "no baseline checkpoint found"
print("weights:", weights)

## After the run

Save Version to keep `runs/` and `reports/`. The full-data result is written to `SUMMARY.md`.